# Exercise solutions — CNN image classification

**DeepVision Masterclass · Solutions companion**

These are worked reference solutions for the exercises in the matching numbered lesson. Run the parent lesson first when variables or datasets are reused. Dataset-specific experiments are left as clearly marked tasks; no performance scores are fabricated.

## Exercise 1 — Residual block
A residual block learns a residual mapping and adds an identity/1×1 projection shortcut.

In [ ]:
import torch
from torch import nn
class ResidualBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.body=nn.Sequential(nn.Conv2d(cin,cout,3,stride,1,bias=False),nn.BatchNorm2d(cout),nn.ReLU(inplace=True),nn.Conv2d(cout,cout,3,1,1,bias=False),nn.BatchNorm2d(cout))
        self.skip=nn.Identity() if cin==cout and stride==1 else nn.Sequential(nn.Conv2d(cin,cout,1,stride,bias=False),nn.BatchNorm2d(cout))
        self.act=nn.ReLU(inplace=True)
    def forward(self,x): return self.act(self.body(x)+self.skip(x))
block=ResidualBlock(32,64,stride=2)
print(block(torch.randn(2,32,32,32)).shape)

## Exercise 2 — SGD with momentum vs AdamW
Rebuild the model from the same initialization for each run. Use a validation-driven scheduler and keep the test set for the final single evaluation.

In [ ]:
optimizer_sgd=torch.optim.SGD(model.parameters(),lr=.05,momentum=.9,weight_decay=5e-4,nesterov=True)
# Alternative optimizer after reinitializing model:
optimizer_adamw=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)

## Exercise 3 — Cosine schedule and early stopping
This helper stops when validation loss has not improved for `patience` epochs.

In [ ]:
import copy, math
# Attach after creating optimizer; call scheduler.step() once per epoch.
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
best_loss=float('inf'); best_state=None; patience=4; bad_epochs=0
# Inside epoch loop, after computing val_loss:
# scheduler.step()
# if val_loss < best_loss:
#     best_loss=val_loss; best_state=copy.deepcopy(model.state_dict()); bad_epochs=0
# else:
#     bad_epochs += 1
# if bad_epochs >= patience: break
# if best_state is not None: model.load_state_dict(best_state)

## Exercise 4 — Macro-F1 and per-class recall
Use predictions from the validation or test evaluation routine.

In [ ]:
from sklearn.metrics import classification_report, f1_score, recall_score
# y_true and y_pred are integer arrays from evaluate().
print('Macro F1:',f1_score(y_true,y_pred,average='macro'))
print('Macro recall:',recall_score(y_true,y_pred,average='macro'))
print(classification_report(y_true,y_pred,target_names=classes,digits=4))

## Exercise 5 — Augmentation ablation
Train two independently initialized models using identical splits and optimization. Change only the training transform: baseline `ToTensor()+Normalize` vs `RandomCrop+RandomHorizontalFlip+Normalize`. Report multiple seeds.

## Reporting template
Record the dataset/source and license, split strategy, seed(s), preprocessing, model configuration, optimizer, epochs, best validation checkpoint, final test metrics, hardware and runtime. Replace illustrative code comments with measured results from your own run.